# PyMIF beginner notebook 06 - append labels inside an existing Zarr

This is the safest beginner workflow for segmentation labels:

1. Open an existing image Zarr with `mode="r+"`.
2. Create explicit label metadata with `axes="tzyx"` or another no-channel axis order.
3. Call `create_empty_group("label_name", label_metadata, is_label=True)`.
4. Write data using `write_label_region(..., group="labels/label_name")`.
5. Re-read the Zarr and inspect `z.labels`.

Important: most label images are integer arrays and should use `data_type="label"`.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

In [ ]:
def make_small_raw_manager(seed=0, num_levels=2):
    """Create a tiny TCZYX intensity dataset for examples."""
    rng = np.random.default_rng(seed)
    raw = rng.integers(0, 1200, size=(2, 3, 4, 64, 64), dtype=np.uint16)
    metadata = {
        "name": "small_raw",
        "axes": "tczyx",
        "size": [raw.shape],
        "chunksize": [(1, 1, 2, 32, 32)],
        "scales": [(2.0, 0.30, 0.30)],   # z, y, x spacing because axes contains zyx
        "units": ("micrometer", "micrometer", "micrometer"),
        "time_increment": 60.0,
        "time_increment_unit": "second",
        "channel_names": ["DAPI", "GFP", "RFP"],
        "channel_colors": ["0000FF", "00FF00", "FF0000"],
        "dtype": "uint16",
        "data_type": "intensity",
    }
    manager = mm.ArrayManager(raw, metadata, chunks=metadata["chunksize"][0])
    if num_levels > 1:
        manager.build_pyramid(num_levels=num_levels, downscale_factor=(1, 2, 2))
    return manager

In [ ]:
def label_metadata_from_image_metadata(image_metadata, name="nuclei", dtype="uint32"):
    """Make safe label metadata from image metadata by removing the channel axis.

    Most segmentation labels are one integer label image per time point and do
    not have an image channel axis. This helper avoids confusion from legacy
    TCZYX examples by creating explicit TZYX label metadata.
    """
    md = dict(image_metadata)
    axes = str(md["axes"]).lower()
    if "c" in axes:
        c_axis = axes.index("c")
        md["axes"] = "".join(ax for ax in axes if ax != "c")
        md["size"] = [tuple(v for i, v in enumerate(size) if i != c_axis) for size in md["size"]]
        md["chunksize"] = [tuple(v for i, v in enumerate(chunks) if i != c_axis) for chunks in md["chunksize"]]
    md["name"] = name
    md["dtype"] = dtype
    md["data_type"] = "label"
    md["is_label"] = True
    md["channel_names"] = []
    md["channel_colors"] = []
    return md

## Start with an existing raw image Zarr

In [ ]:
raw_manager = make_small_raw_manager(seed=20, num_levels=2)
path = clean_path(OUTPUT_DIR / "append_labels_existing.zarr")
raw_manager.to_zarr(path, ngff_version="0.5", zarr_format=3)
z = mm.ZarrManager(path, mode="r+")
print("Existing labels:", list(z.labels.keys()))

## Make explicit label metadata

This helper removes the image channel axis from TCZYX metadata and creates TZYX label metadata. That is the least confusing pattern for beginners.

In [ ]:
label_md = label_metadata_from_image_metadata(z.metadata, name="nuclei", dtype="uint32")
print(label_md["axes"])
print(label_md["size"])
print(label_md["chunksize"])
print(label_md["data_type"])

## Create an empty label group under `/labels/nuclei`

Use just the short name (`"nuclei"`) when creating the group with `is_label=True`.

In [ ]:
nuclei = z.create_empty_group("nuclei", label_md, is_label=True)
print(nuclei)
print("Labels after creation:", list(z.labels.keys()))

## Write label data

Use the full group path (`"labels/nuclei"`) when writing. The label array shape here is TZYX.

In [ ]:
labels = np.zeros(label_md["size"][0], dtype=np.uint32)
labels[:, :, 10:30, 10:30] = 1
labels[:, :, 35:55, 20:45] = 2
labels[:, :, 20:40, 48:60] = 3

z.write_label_region(
    labels,
    group="labels/nuclei",
    level=0,
    downscale_factor=(1, 2, 2),
)
z.read()
print("Labels after writing:", list(z.labels.keys()))
print(z.labels["nuclei"])

## Append another label group

A single image Zarr can contain multiple label groups, for example `nuclei`, `cells`, and `tissue_regions`.

In [ ]:
cell_md = label_metadata_from_image_metadata(z.metadata, name="cells", dtype="uint32")
z.create_empty_group("cells", cell_md, is_label=True)

cells = np.zeros(cell_md["size"][0], dtype=np.uint32)
cells[:, :, 5:50, 5:50] = 10
cells[:, :, 30:62, 30:62] = 11
z.write_label_region(cells, group="labels/cells", level=0, downscale_factor=(1, 2, 2))
z.read()
print("All labels:", list(z.labels.keys()))

## Verify small regions

In [ ]:
for name in z.labels:
    arr = z.labels[name].data[0]
    print(name, "shape:", arr.shape, "dtype:", arr.dtype, "unique small sample:", np.unique(arr[0, 0, :40, :40].compute()))

## Save a copy with labels included

This is useful after adding labels to an existing Zarr and wanting a clean final output path.

In [ ]:
final_path = clean_path(OUTPUT_DIR / "append_labels_existing_final_copy.zarr")
z.to_zarr(final_path, include_groups=True, include_labels=True, ngff_version="0.5", zarr_format=3)
final_z = mm.ZarrManager(final_path, mode="r")
print("Final labels:", list(final_z.labels.keys()))